# Net worth over time

Reads the latest run file for the current scenario under `data/shared/state/runs/` and plots month-end net worth (nominal) plus an inflation-adjusted (CPI) series.

The run file is opened read-only via `RunConnection`; all reads go through `rc.items(...)`, which returns tidy DataFrames suitable for pandas groupby / pivot. See [ADR-002](../docs/architecture/ADR-002-storage-schema.md) for the underlying schema.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.

> **Chart or table clipped, or the notebook slow?** Run `analysis jupyter` and open this notebook in JupyterLab, which can use less memory than VS Code. `analysis sim explain <runfile>` prints the year-end net worth in the terminal, and `analysis sim report income|expenses|gains|tax` the sections below.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.types import DollarMode
from analysis.sim.notebook_ui.pickers import (
    scenario_picker, AfterTaxPicker, DollarModePicker, TimeRangePicker,
)
from analysis.sim.notebook_ui.networth_plots import (
    net_worth_widget, income_widget, expenses_widget, economic_income_widget,
    gains_widget, guard_rails_widget, spending_widget, tax_widget,
)

DOLLAR_MODE = DollarModePicker(dollar_mode=DollarMode.CPI_CONSTANT)
# The years every chart and table below shows; it starts at the whole run.
TIME_RANGE = TimeRangePicker()
# After-tax or pre-tax, for net worth and its change alike.
AFTER_TAX = AfterTaxPicker()

# If new scenarios are added, rerun the notebook to pick them up.
# CONNECTIONS holds the live ScenarioConnection/RunConnection pair; it's
# refreshed in place whenever the dropdown changes.
PICKER, CONNECTIONS, SCENARIO_PICKER = scenario_picker(default="base")
display(PICKER)


## Net worth over time

The **Net worth** toggle starts on **After-tax**: net worth less the tax still owed on it, which is what the money is worth to spend. The table shows the two amounts taken off:

- **deferred_tax** is the tax a traditional IRA or 401(k) balance, and the unrealized gain in a brokerage lot, would owe when spent. The run prices it as if that money were drawn down evenly over 10 years, each year's share stacked on that year's other income, at that year's brackets, filing status and Social Security taxation. So the rate follows the year: higher while salaries are coming in, lower in a quiet year between retirement and Social Security.
- **unpaid_tax** is a finished year's tax still due, from December 31 until the April payment posts.

Because the rate uses each year's filing status, expect a step down the year after a spouse's death, when the survivor files Single, with brackets about half as wide and half the standard deduction (the "widow's penalty"). Converting to Roth while still filing jointly is the usual answer to it. At the death itself, the gains in a brokerage lot or a home mostly disappear: inherited property takes its value at death as its cost basis, so the deferred tax on those gains goes too. Under `"community_property": true` in `scenario.csv`'s `meta_json` (California and the other community-property states) a joint lot's whole gain goes at the first death; otherwise half of it, and the rest at the second. A dashed red line on the charts marks each death; the legends leave it out.

`meta_json` can also change the assumption: `{"after_tax": {"drawdown_years": 15}}` spreads the drawdown wider, and `"step_up": true` leaves out all the gains, as if everything had just stepped up. **Pre-tax** shows the plain total of the balances.

In [ ]:
CONNECTIONS.show(net_worth_widget, time_range=TIME_RANGE, after_tax=AFTER_TAX)

## Income by category

The categories of `sim report income` (its chart is `sim report income --graph`). Income flows are transfers _out of_ accounts with `kind = 'income'`, bucketed by subkind:

- **salary** — `salary`
- **soc security** — `ss_benefit`
- **investment** — `interest`, `dividend`, `capital_gain`, and the growth of what stays invested (a house included), less market drops and a vehicle's depreciation
- **business** — a business account's income less its expenses (the Schedule C net)
- **other** — `pension`, `rental`, `other_income` and the rest

A category with nothing in any year is left out.


In [ ]:
# Partial-first-year catch-up (salary/investment/SS already received before
# scenario.start) is authored as scenario data (ytd_amount/ytd_preroll on
# scheduled.csv, or year_to_date.csv), so it's already included below --
# the engine posts it as ordinary transfer_event rows (ADR-003 §4.11).
CONNECTIONS.show(income_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)


## Expenses by category

The categories of `sim report expenses`. Expense flows are transfers _into_ accounts with `kind = 'expense'`. Subkinds are bucketed:

- **housing** — `housing`, `interest_expense`
- **healthcare** — `healthcare`
- **taxes** — `taxes_federal`, `taxes_state`, `taxes_property`
- **living** — `living`
- **other** — `other_expense`

The built-in `depreciation` account is excluded: it's a book entry recording value lost by wasting assets (vehicles), not money spent, so counting it here would inflate expenses. *Economic income and net worth change* below counts it as an investment loss, like a market drop.


In [ ]:
CONNECTIONS.show(expenses_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

## Spending against the plan

What spending strategies delivered against the budget their rules set, as `sim report spending` shows it (`ADR-010` §6.2). A strategy owns the scheduled rules carrying its `rules` tag and moves their level each year; **delivered** is what was spent (what the expenses above count), the dashed **plan** what the rules would have spent. Without a spending strategy the two are the same.

The table puts the level first (budgeted, delivered, their difference and `of_plan`), then the shape: **redirected** is the share of the year's spending that went somewhere other than the plan's mix, and `filled:GROUP` each changed group's delivered over budgeted. **By** groups by category, account or a tag role, as `sim report expenses --by` does.

In [ ]:
CONNECTIONS.show(spending_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

## Realized vs. unrealized gains

Splits every income-side flow into three buckets, to check whether actual cash distributions (not paper growth) cover expenses:

- **taxable** — reaches a liquid account (`cash` subkind) with a taxable character (`ordinary`, `ltcg`, `tax_deferred_out`), plus the *taxable portion* of Social Security.
- **non-taxable** — reaches a liquid account with a `tax_free` character (Roth growth/withdrawals, tax-free income), plus the *non-taxable portion* of Social Security.
- **unrealized** — income-kind growth (interest, dividends, appreciation) that stays inside a retirement/brokerage account and never reaches cash.

Social Security is **not** simply lumped into "taxable": only up to 85% of it ever is (per the federal provisional-income test — see `_taxable_social_security_cents`), and it can be 0% at low enough income. We split each year's SS benefit using the engine's own computed `federal_taxable_ss` (from `tax_year_summary`), not a flat assumption.

Social Security is also counted **gross**: Medicare Part B/D premiums are withheld before the deposit, so those legs go straight from the SS income account to a healthcare expense account and never touch cash. They still count as received (and taxable) income, so `ss_benefit` is bucketed on tax character alone rather than on reaching a liquid account.

Caveats (this is a first pass — flag anything that doesn't match your mental model):

- `return_of_capital` (basis returned on a sale) and `tax_deferred_in` (contributions) are excluded entirely: they're principal movements, not gains.
- "Liquid" = the `cash` subkind only; a distribution landing in, say, an HSA cash account wouldn't count as realized here.
- The scenario's final year has no computed `federal_taxable_ss` yet (reconciliation happens the following April), so that year's SS is left entirely in "non-taxable" — an understatement, consistent with the same caveat in "After-tax income" below.
- `distribution_coverage` below is `taxable + non-taxable − expenses`; negative years are ones where realized cash distributions alone didn't cover spending (the gap was plugged by `waterfall_fill`/`refill_transfer` principal draws, which aren't gains and so aren't counted as coverage here). Medicare premiums withheld from Social Security are in both sides of that subtraction, as they should be.


In [ ]:
CONNECTIONS.show(gains_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

## Economic income and net worth change

Each year's whole story, the same columns as `sim report income`. **Economic income** is income plus paper gains, net of losses: what the year added before spending, gains counted as they accrue whether or not anything was sold (economists call it Haig-Simons income). Its `investment` part is the growth of everything invested, a house included, less market drops and a vehicle's depreciation. It is carried down to the change in net worth, and each row adds up:

- **after_tax_income** = economic income − **tax** − **Δdeferred_tax**. Tax is the year's federal and state liability, attributed to the year the income was earned, whatever year it is paid or refunded. The deferred line is the change in tax still owed on traditional IRA and 401(k) balances and unrealized gains (see *Net worth over time*): growth inside an IRA isn't yours to spend until its tax is paid.
- **Δnet_worth** = after-tax income − **spending** − **inflation**. Spending is every expense but income tax and business expenses, which come off business income (its Schedule C net) instead. Inflation, in CPI-constant dollars only, is what inflation took from the real value of what was held: money sitting still loses value.

On **Pre-tax** the tax line is **tax_paid**, what was paid in the year (mostly the previous year's balance due and this year's withholding), there is no deferred line, and net worth is the plain total of the balances.

The first year runs from its January 1, worked back from the opening balances and the year-to-date rows dated before the scenario starts.

> **Tax timing:** the table under the chart reconciles each tax year: its liability, what was withheld or paid during it, and the balance due or refund settled the April after. Do not read the reconciliation year as the tax year.

The engine runs a hidden tax-only tail after the declared scenario end, so the final scenario year has a computed tax liability without displaying a later simulation year.

In [ ]:
CONNECTIONS.show(economic_income_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE,
                 after_tax=AFTER_TAX)

## Tax

Federal and state tax by year, as `sim report tax` shows it (`--graph` draws the same chart): the tax stacked, with the **effective rate** (total tax over federal AGI) on the right-hand axis. The table gives, for each of federal (`fed_`) and state (`state_`, when the scenario computes it), the AGI, the deduction taken, the taxable income its brackets taxed (`income`), and the tax. Tax belongs to the year its income was earned in, though it is settled the April after: the *Economic income* section's second table shows each year's reconciliation.


In [ ]:
CONNECTIONS.show(tax_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)

## Guard rails

Where the run stands against the scenario's guard rails (`guard_rails.csv`): lines its spending, or its withdrawals from the portfolio, should stay on one side of, such as the 4% rule or Guyton-Klinger's rails (see `docs/guides/guard-rails.md`).

Each measure gets a column: on top, the year's measure against each rail's threshold, shaded where it is past the rail; below, each rail's **cumulative margin**, the years since the rail began added up: room banked above zero, overdrawn below. The top answers "is this year's rate a problem?", the bottom "is it spending room left from earlier years, or still repaying an overdraft?".

A rail to stay inside is solid and warm; a **positive** rail (past it is good news, such as room to spend more) is dashed, cool and hatched. The dotted lines carry the cumulative margin at the portfolio's own return: an overdraft costs more than its sum, since the money spent would have kept growing. Both are guides, not a re-run.

Rails are read from the scenario when shown, so an edited rail needs no new run, only this cell run again.

In [ ]:
CONNECTIONS.show(guard_rails_widget, dollar_mode=DOLLAR_MODE, time_range=TIME_RANGE)